In [ ]:
# In this file, we set up a jsonl file to record model's choice of each question.
# We use gpt API to extract final choice from full-output answers.

In [ ]:
import json

def create_outcome_jsonl():
    input_file = "/SFT/Qwen3-0.6B/testbench/mcQTSA_test_final.jsonl"
    output_file = "Outcome_1000.jsonl"
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f_in:
            lines = f_in.readlines()

        if len(lines) < 1000:
            print(f"警告：输入文件只有 {len(lines)} 行，但需要100行")
            lines = lines[:1000] 
        
        with open(output_file, 'w', encoding='utf-8') as f_out:
            for i in range(1000):
                if i < len(lines):
                
                    original_data = json.loads(lines[i].strip())
                    answer = original_data.get("answer", "")
                else:
                    answer = ""

                new_data = {
                    "ID": str(i + 1),  
                    "Correct Answer": answer
                }

                f_out.write(json.dumps(new_data, ensure_ascii=False) + '\n')
        
        print(f"Finish set up {output_file} file，including {min(1000, len(lines))} data")
        
    except FileNotFoundError:
        print(f"错误：找不到输入文件 {input_file}")
    except json.JSONDecodeError as e:
        print(f"错误：JSON解析失败 - {e}")
    except Exception as e:
        print(f"错误：{e}")

if __name__ == "__main__":
    create_outcome_jsonl()

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-1B/1000QTSA_test/(new)1000_1B_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)1B_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-1B-Instruct/1000QTSA_test/(new)1000_1B_Instruct_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)1B_Instruct_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-3B/1000QTSA_test/(new)1000_3B_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)3B_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-3B-Instruct/1000QTSA_test/(new)1000_3B_Instruct_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)3B_Instruct_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-1B-Instruct/1000QTSA_test/(new)1000_finetuned_1B_Instruct_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)finetuned_1B_Instruct_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-1B/1000QTSA_test/(new)1000_finetuned_1B_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)finetuned_1B_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-3B-Instruct/1000QTSA_test/(new)1000_finetuned_3B_Instruct_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)finetuned_3B_Instruct_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")

In [ ]:
import json
import re
import os
from openai import OpenAI

# --- Setup OpenAI client ---
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

input_file_path = "/SFT/Llama3.2-3B/1000QTSA_test/(new)1000_finetuned_3B_prompt2.jsonl"
outcome_file_path = "Outcome_1000.jsonl"

def extract_answer_from_output(question, output_text):
    """Use DeepSeek API to extract answer options from question and output text"""
    system_prompt = """You are an answer extraction specialist. Your task is to extract the multiple-choice answer option(s) that the output text is selecting or indicating.
    
IMPORTANT GUIDELINES:
1. ONLY extract what answer the output text is choosing, NOT what the correct answer should be
2. Do NOT evaluate whether the output's choice is correct or not
3. Do NOT add your own reasoning or interpretation
4. Focus solely on identifying which option(s) the output text is selecting
5. If the output expresses uncertainty but still makes a choice, extract that choice
6. If multiple options are selected in the output, extract all of them

Return format:
- For single answers: return just the letter (e.g., "A")
- For multiple answers: return the letters concatenated without any separators (e.g., "BCD" for options B, C, D)
- Only return the letter(s), no explanations, no punctuation, no extra text.
- If the answer is strange and doesn't answer any choice, please return None."""
    
    user_prompt = f"""
Question: "{question}"

Output text: "{output_text}"

Based on the output text above, extract the multiple-choice answer option(s) that the output is selecting. Return ONLY the letter(s) in standardized format.

Standardized answer:
"""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4.1-nano",  
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=10,
            temperature=0.1,
            stream=False
        )
        
        answer = response.choices[0].message.content.strip()
        
        # Use regex to extract answer options (A, B, C, D, etc.)
        match = re.search(r'[A-D]+', answer)
        if match:
            return match.group()
        return None
    except Exception as e:
        print(f"API call error: {e}")
        return None

input_data = []
with open(input_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        input_data.append(json.loads(line.strip()))

outcome_data = []
with open(outcome_file_path, 'r', encoding='utf-8') as f:
    for line in f:
        outcome_data.append(json.loads(line.strip()))

max_items = 1000
processed_count = 0

for i, item in enumerate(input_data):
    if i >= len(outcome_data) or processed_count >= max_items:
        break
        
    question = item.get('question', '')
    output_text = item.get('output', '')
    answer = extract_answer_from_output(question, output_text)

    outcome_data[i]["(new)finetuned_3B_prompt2 Answer"] = answer
    print(f"ID {outcome_data[i]['ID']}: Extracted answer is {answer}")
    
    processed_count += 1

with open(outcome_file_path, 'w', encoding='utf-8') as f:
    for item in outcome_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("Processing completed!")